> 在线讲义：[第 22 章 · KV cache 与 MQA / GQA：生成时为什么要存 K、V，怎么让它变小](https://wikieden.github.io/beyond-transformer/lessons/22-kv-cache-gqa.html) · 本书《超越 Transformer》

# 第 22 章 · KV cache 与 MQA / GQA：生成时为什么要存 K、V，怎么让它变小

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/22-kv-cache-gqa.html) · 仓库里 `beyond-transformer/notes/lessons/22-kv-cache-gqa.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 cache vs 从头算 | 投影次数 | `generate(x, use_cache=...)` | 6 vs 21，输出一样 |
| 第二步 cache 大小 | 2·L·H_kv·d·n·字节 | `cache_bytes(...)` | 2.000 / 0.500 / 0.062 GiB |
| 第三步 GQA | 分组、取平均 | `attend_last`、`mean_pool_heads` | (8, 8) → (8, 4) |

带 cache 和每步从头算结果一样，但计算量从 O(n²) 降到 O(n)；cache 的字节数 2·L·H_kv·d_head·n·字节；
几个 Q 头共用一组 K、V（GQA），cache 缩小 H / H_kv 倍。
在 a1 模型上写 KV cache 解码、加 GQA 并实测 cache 大小是里程碑 T-M5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
D, H = 8, 4
DH = D // H

**函数 `weights`**：小工具。

In [ ]:
def weights(seed=0, n_kv=H):
    rng = np.random.default_rng(seed)
    return rng.normal(0, 0.5, (D, D)), rng.normal(0, 0.5, (D, n_kv * DH)), rng.normal(0, 0.5, (D, n_kv * DH))

**函数 `attend_last`**：一个 query（H 个头）对 K、V（n × H_kv × d_head）做注意力；GQA 时第 h 个 Q 头用第 h // (H / H_kv) 组 K、V。

In [ ]:
def attend_last(q, K, V):
    """一个 query（H 个头）对 K、V（n × H_kv × d_head）做注意力；GQA 时第 h 个 Q 头用第 h // (H / H_kv) 组 K、V。"""
    n_kv = K.shape[1]
    g = H // n_kv
    out = []
    for h in range(H):
        k, v = K[:, h // g], V[:, h // g]
        s = k @ q[h] / np.sqrt(DH)
        w = np.exp(s - s.max())
        w /= w.sum()
        out.append(w @ v)
    return np.concatenate(out)

**函数 `generate`**：x：n × D 的输入（已经是每步的隐状态）。返回每一步最后一个位置的注意力输出和做了多少次 K、V 投影。

In [ ]:
def generate(x, n_kv=H, use_cache=True):
    """x：n × D 的输入（已经是每步的隐状态）。返回每一步最后一个位置的注意力输出和做了多少次 K、V 投影。"""
    Wq, Wk, Wv = weights(n_kv=n_kv)
    outs, proj = [], 0
    Kc, Vc = [], []
    for t in range(len(x)):
        q = (x[t] @ Wq).reshape(H, DH)
        if use_cache:
            Kc.append((x[t] @ Wk).reshape(n_kv, DH))
            Vc.append((x[t] @ Wv).reshape(n_kv, DH))
            proj += 1
            K, V = np.array(Kc), np.array(Vc)
        else:
            K = (x[: t + 1] @ Wk).reshape(t + 1, n_kv, DH)
            V = (x[: t + 1] @ Wv).reshape(t + 1, n_kv, DH)
            proj += t + 1
        outs.append(attend_last(q, K, V))
    return np.array(outs), proj

**函数 `mean_pool_heads`**：GQA 论文 §2.1：把 MHA 的 H 个 K（或 V）头按组取平均，得到 H_kv 个头，再少量继续训练（uptraining）。

In [ ]:
def mean_pool_heads(W, n_kv):
    """GQA 论文 §2.1：把 MHA 的 H 个 K（或 V）头按组取平均，得到 H_kv 个头，再少量继续训练（uptraining）。"""
    d, hd = W.shape
    heads = W.reshape(d, H, hd // H)
    return heads.reshape(d, n_kv, H // n_kv, hd // H).mean(2).reshape(d, n_kv * (hd // H))

**函数 `cache_bytes`**：K 和 V 两份 × 层数 × KV 头数 × 每头维数 × token 数 × 每个数的字节数。

In [ ]:
def cache_bytes(layers, n_kv, d_head, n_tokens, bytes_per=2, batch=1):
    """K 和 V 两份 × 层数 × KV 头数 × 每头维数 × token 数 × 每个数的字节数。"""
    return 2 * layers * n_kv * d_head * n_tokens * bytes_per * batch

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
x = np.random.default_rng(1).normal(size=(6, D))
print("第一步 带 cache 和从头算，结果一样")
a, pa = generate(x, use_cache=True)
b, pb = generate(x, use_cache=False)
print(f"  6 步生成：最大差 {np.abs(a - b).max():.1e}；K、V 投影次数：带 cache {pa}，从头算 {pb}（= 1 + 2 + … + 6）")
print(f"  生成 1000 步：带 cache 1000 次，从头算 {1000 * 1001 // 2:,} 次")
print("\n第二步 cache 有多大（32 层、32 个 Q 头、d_head = 128、bf16）")
for name, nkv in (("MHA（32 组 K、V）", 32), ("GQA（8 组）", 8), ("MQA（1 组）", 1)):
    gb = cache_bytes(32, nkv, 128, 4096) / 2**30
    print(f"  {name}：4096 个 token 的 cache {gb:.3f} GiB；batch 32 时 {gb * 32:.1f} GiB")

In [ ]:
print("\n第三步 GQA：几个 Q 头共用一组 K、V")
for nkv in (4, 2, 1):
    print(f"  H = 4、H_kv = {nkv}：Q 头 → K/V 组 {[h // (H // nkv) for h in range(H)]}，cache 是 MHA 的 1/{H // nkv}")
_, Wk, _ = weights()
pooled = mean_pool_heads(Wk, 2)
print(f"  从 MHA 改成 GQA（uptraining 的初始化）：W_K {Wk.shape} → 组内取平均 {pooled.shape}")

## 3. 代码里才看得到的

- **cache 存的是投影之后的 K、V**：不是隐状态 x；存 x 每步还得重算投影，存 K、V 才真正省计算（第 25 章的 MLA 反过来存一个压缩向量，用吸收的技巧避免展开）。
- **RoPE 要在存进 cache 之前转好**：k 按它自己的位置转完再存；新 token 的位置等于当前 cache 长度。这里的玩具没有位置编码，看不到这一点。

## 4. 自己改一改

- 把 `cache_bytes` 的每数字节改成 1（fp8 cache），MHA batch 32 的 cache 变成多少？
- 把 `mean_pool_heads` 改成“只取组里第一个头”而不是平均，你觉得 uptrain 起来会更好还是更差？（GQA 论文的消融里有答案）

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 写过的字记草稿本：因果下前面的 K、V 不变，存起来每步只算新的；结果一样，n 方变 n。
> cache 是二乘层乘组乘头维乘长度乘字节，长了、批大了比权重还大；解码每步读一遍，卡的是带宽。
> 所有头共用一组是 MQA，几个头一组是 GQA；组内取平均，再训百分之五就改过来。